# 04 — Modeling (Phase 2 — Exploratory Expansion with NOAA Intensity Features)

## Scope reminder
Phase 2 is an **exploratory expansion** scoped to hydrometeorological events (~80% of FEMA PA
declarations). The Phase 1 comparison below uses Phase 1's results on the full dataset —
this is intentional and transparently flagged. The goal is to show that adding physical
intensity data improves performance on the event types where it exists, not to claim
Phase 2 replaces Phase 1 for all disaster types.

## Feature set comparison vs Phase 1

| Feature group        | Phase 1 | Phase 2 |
|----------------------|---------|---------|
| incidentType         | ✓       | ✓       |
| stateAbbreviation    | ✓       | ✓       |
| incident_season      | ✓       | ✓       |
| declaration_lag_days | ✓       | ✓       |
| incident_duration    | ✓       | ✓       |
| n_counties           | ✓       | ✓       |
| prior_disasters_5yr  | ✓       | ✓       |
| population           | ✓       | ✓       |
| median_income        | ✓       | ✓       |
| poverty_rate         | ✓       | ✓       |
| risk_score (NRI)     | ✓       | ✓       |
| noaa_peak_magnitude  | ✗       | ✓       |
| noaa_log_damage      | ✗       | ✓       |
| noaa_log_deaths      | ✗       | ✓       |
| noaa_log_injuries    | ✗       | ✓       |
| noaa_event_count     | ✗       | ✓       |
| noaa_matched         | ✗       | ✓       |

## Hypothesis
Adding event-level physical intensity (NOAA) should primarily improve:
- **Major (Tier 2) recall** — the tier most sensitive to intensity variance
- **Overall F1** — especially for disasters where NOAA matched (`noaa_matched=1`)

## Note on Phase 1 comparison
Phase 1 F1=0.690 was trained on ALL disaster types. Phase 2 trains on hydrometeorological
events only (~80% of data). A delta > 0 indicates NOAA features add signal beyond what
was available in Phase 1 for this subset.

In [ ]:
import pandas as pd
import numpy as np
import os, sys, warnings, pickle
warnings.filterwarnings('ignore')

sys.path.insert(0, '../../')
from utils import classification_metrics, DISASTER_LABELS

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

PROCESSED = '../data/processed/'

PHASE1_TEST_F1 = 0.6903  # Logistic Regression Phase 1 baseline for comparison

## Define Feature Sets
Phase 2 feature set extends Phase 1 with 6 new NOAA-derived columns.

In [ ]:
CAT_FEATURES = ['incidentType', 'stateAbbreviation', 'incident_season']
NUM_FEATURES = [
    # Phase 1 features
    'declaration_lag_days', 'incident_duration_days', 'n_counties',
    'prior_disasters_5yr', 'population', 'median_income', 'poverty_rate', 'risk_score',
    # Phase 2 NOAA features
    'noaa_peak_magnitude', 'noaa_log_damage', 'noaa_log_deaths',
    'noaa_log_injuries', 'noaa_event_count', 'noaa_matched'
]
TARGET          = 'funding_tier'
SPLIT_YEAR      = 2018
VALIDATION_YEAR = 2016
TARGET_NAMES    = [DISASTER_LABELS[i] for i in range(3)]

## Load Data and Create Train/Val/Test Splits
We use a temporal split identical to Phase 1 to ensure comparability:
- Train: before 2016
- Validation: 2016–2017
- Test: 2018 onwards

In [ ]:
df = pd.read_csv(PROCESSED + 'cleaned_fema_noaa.csv', low_memory=False)
df['incidentBeginDate'] = pd.to_datetime(df['incidentBeginDate'], errors='coerce')
df['incident_year']     = df['incidentBeginDate'].dt.year

df_model = df[df[TARGET].notna()].copy()

FEATURES = CAT_FEATURES + NUM_FEATURES
for col in NUM_FEATURES:
    if col not in df_model.columns:
        df_model[col] = 0

train = df_model[df_model['incident_year'] <  VALIDATION_YEAR]
val   = df_model[(df_model['incident_year'] >= VALIDATION_YEAR) & (df_model['incident_year'] < SPLIT_YEAR)]
test  = df_model[df_model['incident_year'] >= SPLIT_YEAR]

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')
print(f'NOAA match rate — train: {train["noaa_matched"].mean():.1%} | test: {test["noaa_matched"].mean():.1%}')

## Build Preprocessing Pipeline
Categorical features: impute with mode, one-hot encode.
Numeric features: impute with median, standard scale.

In [ ]:
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
num_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler())
])
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, CAT_FEATURES),
    ('num', num_pipe, NUM_FEATURES)
])

## Train All 4 Models
Same model set as Phase 1 for direct comparison. Class weights balanced to address
the class imbalance between Minor (many) and Major (few) tiers.

In [ ]:
MODELS = {
    'Baseline (Stratified)': DummyClassifier(strategy='stratified', random_state=42),
    'Logistic Regression':   LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=200, random_state=42),
}

results_val  = {}
results_test = {}

for name, model in MODELS.items():
    pipe = Pipeline([('pre', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    
    vp = pipe.predict(X_val)
    tp = pipe.predict(X_test)
    
    vf1 = f1_score(y_val,  vp, average='weighted', zero_division=0)
    tf1 = f1_score(y_test, tp, average='weighted', zero_division=0)
    
    results_val[name]  = {'F1_weighted': vf1, 'pipeline': pipe}
    results_test[name] = {'F1_weighted': tf1, 'preds': tp}
    
    print(f'\n{"="*54}')
    print(f'  {name}')
    print(f'{"="*54}')
    print(f'  Val  F1: {vf1:.4f}')
    print(classification_report(y_test, tp, target_names=TARGET_NAMES, zero_division=0))

## Hyperparameter Tuning
Tune the best-performing non-baseline model using RandomizedSearchCV with TimeSeriesSplit
to respect temporal ordering.

In [ ]:
best_name = max(results_val, key=lambda k: results_val[k]['F1_weighted']
                if k != 'Baseline (Stratified)' else 0)
print(f'Best model for tuning: {best_name}')

param_grid = {
    'model__n_estimators':      [100, 200, 300],
    'model__max_depth':         [3, 4, 5],
    'model__learning_rate':     [0.05, 0.1, 0.15],
    'model__subsample':         [0.6, 0.7, 0.8],
    'model__min_samples_split': [5, 10, 20, 40],
    'model__min_samples_leaf':  [5, 10, 20],
} if 'Boosting' in best_name else {
    'model__C':          [0.01, 0.1, 1, 10],
    'model__solver':     ['lbfgs','saga'],
    'model__max_iter':   [500, 1000],
}

best_pipe = Pipeline([
    ('pre',   preprocessor),
    ('model', MODELS[best_name])
])

tscv = TimeSeriesSplit(n_splits=5)
search = RandomizedSearchCV(best_pipe, param_grid, n_iter=30, cv=tscv,
                             scoring='f1_weighted', random_state=42, n_jobs=-1, verbose=1)
search.fit(X_train, y_train)

print(f'Best params: {search.best_params_}')
print(f'Best CV F1:  {search.best_score_:.4f}')

tuned_pipe  = search.best_estimator_
tuned_vp    = tuned_pipe.predict(X_val)
tuned_tp    = tuned_pipe.predict(X_test)
tuned_vf1   = f1_score(y_val,  tuned_vp, average='weighted', zero_division=0)
tuned_tf1   = f1_score(y_test, tuned_tp, average='weighted', zero_division=0)

results_val[f'{best_name} (Tuned)']  = {'F1_weighted': tuned_vf1, 'pipeline': tuned_pipe}
results_test[f'{best_name} (Tuned)'] = {'F1_weighted': tuned_tf1, 'preds': tuned_tp}

print(f'\nTuned Val F1:  {tuned_vf1:.4f}')
print(f'Tuned Test F1: {tuned_tf1:.4f}')
print(classification_report(y_test, tuned_tp, target_names=TARGET_NAMES, zero_division=0))

## Phase 1 vs Phase 2 Summary Comparison

In [ ]:
print('\n' + '='*60)
print('  PHASE 1 vs PHASE 2 COMPARISON')
print('='*60)
print(f'{"Model":<35} {"Phase 1 Test F1":>16} {"Phase 2 Test F1":>16} {"Delta":>8}')
print('-'*60)

# Phase 1 known results
p1 = {
    'Baseline (Stratified)': 0.5189,
    'Logistic Regression':   0.6903,
    'Random Forest':         0.5593,
    'Gradient Boosting':     0.6750,
}

for name, res in results_test.items():
    base = name.replace(' (Tuned)', '')
    p1_f1 = p1.get(base, p1.get(name, None))
    p2_f1 = res['F1_weighted']
    delta = f'+{p2_f1 - p1_f1:.4f}' if p1_f1 else 'N/A'
    p1_str = f'{p1_f1:.4f}' if p1_f1 else 'N/A'
    print(f'{name:<35} {p1_str:>16} {p2_f1:>16.4f} {delta:>8}')

## Save Best Model Bundle
Persist the best model pipeline and associated metadata for evaluation in notebook 05.

In [ ]:
best_final = max(results_val, key=lambda k: results_val[k]['F1_weighted'])
best_pipe_final = results_val[best_final]['pipeline']
best_proba = best_pipe_final.predict_proba(X_test)
best_preds = results_test[best_final]['preds']

bundle = {
    'pipeline':       best_pipe_final,
    'X_test':         X_test,
    'y_test':         y_test,
    'preds':          best_preds,
    'proba':          best_proba,
    'y_val':          y_val,
    'val_proba':      best_pipe_final.predict_proba(X_val),
    'val_f1':         results_val[best_final]['F1_weighted'],
    'test_f1':        results_test[best_final]['F1_weighted'],
    'features':       FEATURES,
    'cat_features':   CAT_FEATURES,
    'num_features':   NUM_FEATURES,
    'target_names':   TARGET_NAMES,
    'model_name':     best_final,
    'level':          'disaster_v2',
    'phase1_test_f1': PHASE1_TEST_F1,
}

with open(PROCESSED + 'best_model_v2.pkl', 'wb') as f:
    pickle.dump(bundle, f)
print(f'Saved: best_model_v2.pkl  (model: {best_final})')